# GFlowNet-TB: nhiều seed song song trên 2 GPU

Mỗi GPU có một hàng đợi seed riêng. Với từng seed, notebook chạy Stage 4 TB, Stage 5 fixed-rule và Stage 5 Bayesian-TB, rồi tổng hợp bảng mean±std.

In [ ]:
BACKBONE = 'REPLACE_BACKBONE'
GIT_REF = 'main'
SEEDS = [42, 44, 46, 48, 50]
NO_RESUME_SEEDS = [46, 50]
RUNS = []
MC_SAMPLES = 32
GFLOWNET_ITERATIONS = 5000
NUM_EPOCHS = 100
PATIENCE = 5

In [ ]:
from pathlib import Path

if any(run['dataset_id'] == 'culture-b' for run in RUNS):
    source = Path('/kaggle/input/datasets/utkarshsaxenadn/fast-food-classification-dataset/Fast Food Classification V2')
    for destination, origin in {'train': 'Train', 'test': 'Test', 'val': 'Valid'}.items():
        path = Path('/kaggle/working') / destination
        if path.is_symlink():
            path.unlink()
        elif path.exists():
            raise FileExistsError(f'Không ghi đè path thật: {path}')
        path.symlink_to(source / origin, target_is_directory=True)
for run in RUNS:
    assert Path(run['data_dir']).exists(), run['data_dir']

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/khoaddb2207532/neuro_symbolic_mlops_l2_app.git'
PROJECT = Path('/kaggle/working/neuro_symbolic_mlops_l2_app')
if not PROJECT.exists():
    clone_url = REPO_URL
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('GITHUB_TOKEN')
        if token:
            clone_url = REPO_URL.replace('https://', f'https://{token}@')
    except Exception:
        pass
    subprocess.run(['git', 'clone', '--filter=blob:none', clone_url, str(PROJECT)], check=True)
subprocess.run(['git', 'fetch', 'origin', GIT_REF, '--depth', '1'], cwd=PROJECT, check=True)
subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=PROJECT, check=True)
print('Checked out:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT, text=True).strip())

In [ ]:
%cd /kaggle/working/neuro_symbolic_mlops_l2_app
!pip install -q torchgfn tensordict dvclive dvc openpyxl
!nvidia-smi -L
import torch
assert torch.cuda.device_count() >= 2, 'Hãy chọn Accelerator: GPU T4 x2'
print([torch.cuda.get_device_name(i) for i in range(2)])

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

RUNS_JSON = Path('/kaggle/working/tb_multiseed_runs.json')
RUNS_JSON.write_text(json.dumps(RUNS, indent=2), encoding='utf-8')
OUTPUT = Path('/kaggle/working/tb_multiseed_outputs') / BACKBONE
command = [
    sys.executable, '-m', 'pipelines.run_dual_gpu_tb_multi_seed_experiment',
    '--runs-json', str(RUNS_JSON),
    '--backbone', BACKBONE,
    '--output-dir', str(OUTPUT),
    '--project-dir', str(PROJECT),
    '--kaggle-input-root', '/kaggle/input',
    '--mc-samples', str(MC_SAMPLES),
    '--gflownet-iterations', str(GFLOWNET_ITERATIONS),
    '--num-epochs', str(NUM_EPOCHS),
    '--patience', str(PATIENCE),
    '--no-resume-seeds', *map(str, NO_RESUME_SEEDS),
]
subprocess.run(command, cwd=PROJECT, check=True)

In [ ]:
import pandas as pd

summary = pd.read_csv(OUTPUT / 'tb_all_seed_summary.csv')
display(summary.sort_values(['dataset_id', 'test_f1_macro_mean'], ascending=[True, False]))

## Resume

Save Version rồi Add Input output cũ vào lần chạy kế tiếp. Runner kiểm tra checkpoint từng stage và chỉ chạy phần còn thiếu.